In [1]:

library(Seurat)
library(SeuratDisk)

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.3 but the current
version is 1.7.4; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat



In [2]:
TSmtxdata <- readRDS("/mnt/lscratch/users/adhal/SingleCellUtils/data/scHIIT_ref/TabulaSapiensMatrixSampled.rds")
TSmetdata <- readRDS("/mnt/lscratch/users/adhal/SingleCellUtils/data/scHIIT_ref/TabulaSapiensMetaSampled.rds")

In [3]:
unique(TSmetdata$Annot)

[1] "T.Cell"                     "Macrophage"                
 [3] "Myofibroblast"              "Urothelial.Cell"           
 [5] "Endothelial.Cell"           "Smooth.Muscle.Cell"        
 [7] "Fibroblast"                 "Pericyte"                  
 [9] "Mast.Cell"                  "NK.Cell"                   
[11] "B.Cell"                     "Plasma.Cell"               
[13] "Dendritic.Cell"             "Erythrocyte"               
[15] "Monocyte"                   "Neutrophil"                
[17] "NKT.Cell"                   "Platelet.Cell"             
[19] "Hematopoietic.Stem.Cell"    "Basophil"                  
[21] "Plasmablast"                "Myeloid.Progenitor.Cell"   
[23] "Granulocyte"                "Erythroid.Progenitor.Cell" 
[25] "Epithelial.Cell"            "Microglia"                 
[27] "Photoreceptor.Cell"         "Glial.Cell"                
[29] "Limbal.Stem.Cell"           "Keratocyte"                
[31] "Stromal.Cell"               "Melanocyte"                
[33] "Neuron"                     "Erythroid"                 
[35] "Radial.Glia.Cell"           "Horizontal.Cell"           
[37] "Ciliated.Cell"              "Adipocyte"                 
[39] "Retinal.Ganglion.Cell"      "Leucocyte"                 
[41] "Mesenchymal.Stem.Cell"      "Hepatocyte"                
[43] "Enterocyte"                 "Goblet.Cell"               
[45] "Paneth.Cell"                "Transient.Amplifying.Cell" 
[47] "Intestinal.Crypt.Stem.Cell" "Tuft.Cell"                 
[49] "Monocyte.Macrophage"        "Cholangiocyte"             
[51] "Pneumocyte"                 "Adventitial.Cell"          
[53] "Basal.Cell"                 "Aerocyte"                  
[55] "Club.Cell"                  "Mesothelial.Cell"          
[57] "Ionocyte"                   "Thymocyte"                 
[59] "Lymphoid.Cell"              "Satellite.Cell"            
[61] "Tendon.Cell"                "Fast.Muscle.Cell"          
[63] "Slow.Muscle.Cell"           "Acinar.Cell"               
[65] "Stellate.Cell"              "Ductal.Cell"               
[67] "Beta.Cell"                  "Pp.Cell"                   
[69] "Alpha.Cell"                 "Delta.Cell"                
[71] "Sperm.Cell"                 "Langerhans.Cell"           
[73] "Innate.Lymphoid.Cell"       "Muscle.Cell"               
[75] "Keratinocyte"               "Schwann.Cell"              
[77] "Secretory.Cell"

## Exploring the reference single nuclei data

In [4]:
ref_nuc_norm_data <- readRDS("/mnt/lscratch/users/adhal/scrna_target_idf_v3/data/Ref_sNucSymbol/sNucSymbol_NormData.rds")

In [6]:
ref_nuc_bg_model_pars <- readRDS("/mnt/lscratch/users/adhal/scrna_target_idf_v3/data/Ref_sNucSymbol/sNucSymbol_bg_model_pars.rds")

In [ ]:
# # Convert
# # Create a Seurat object
# seurat_obj <- CreateSeuratObject(counts = ref_nuc_norm_data)

# # Save as h5Seurat first
# SaveH5Seurat(seurat_obj, filename = "ref_nucleus.h5Seurat", overwrite = TRUE)

# # Convert to h5ad
# Convert("ref_nucleus.h5Seurat", dest = "ref_nucleus.h5ad", overwrite = TRUE)

# # Clean up
# file.remove("ref_nucleus.h5Seurat")

Creating h5Seurat file for version 3.1.5.9900

Validating h5Seurat file

Adding data from RNA as X



ERROR: Error in assay.group$obj_copy_to(dst_loc = dfile, dst_name = "X", src_name = x.data): HDF5-API Errors:
    error #000: H5O.c in H5Ocopy(): line 537: unable to synchronously copy object
        class: HDF5
        major: Object header
        minor: Unable to copy object

    error #001: H5O.c in H5O__copy_api_common(): line 447: unable to copy object
        class: HDF5
        major: Object header
        minor: Unable to copy object

    error #002: H5VLcallback.c in H5VL_object_copy(): line 5685: object copy failed
        class: HDF5
        major: Virtual Object Layer
        minor: Unable to copy object

    error #003: H5VLcallback.c in H5VL__object_copy(): line 5646: object copy failed
        class: HDF5
        major: Virtual Object Layer
        minor: Unable to copy object

    error #004: H5VLnative_object.c in H5VL__native_object_copy(): line 155: unable to copy object
        class: HDF5
        major: Object header
        minor: Unable to copy object

    error 

In [12]:
library(Matrix)
# Save as Matrix Market format
writeMM(ref_nuc_norm_data, "matrix.mtx")

# Save row names (genes)
writeLines(rownames(ref_nuc_norm_data), "genes.txt")

# Save column names (cell barcodes)
writeLines(colnames(ref_nuc_norm_data), "barcodes.txt")

NULL

In [ ]:
library(reticulate)
library(Matrix)

# Read your RDS
sparse_matrix <- ref_nuc_norm_data

# Use Python from R
sc <- import("scanpy")
ad <- import("anndata")
scipy_sparse <- import("scipy.sparse")

# Convert to Python sparse matrix
py_matrix <- r_to_py(t(sparse_matrix))  # Transpose for cells x genes

# Create AnnData object
adata <- ad$AnnData(X = py_matrix)
adata$var_names <- rownames(sparse_matrix)
adata$obs_names <- colnames(sparse_matrix)

# Save
adata$write("output.h5ad")